# 05 - Model Comparison

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

saved = joblib.load("../Data/results.pkl")
results, store, y_test = saved["results"], saved["store"], saved["y_test"]

## 1. All Results

In [ ]:
cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "TP", "FP", "FN", "TN"]
results[cols].round(4)

## 2. Tuned Models Ranked by PR-AUC

In [ ]:
tuned_only = results[results.index.str.contains("tuned")].sort_values("PR-AUC", ascending=False)
tuned_only[["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC", "FN", "FP"]].round(4)

In [ ]:
metrics = ["Precision", "Recall", "F1", "PR-AUC"]
tuned_only[metrics].plot(kind="bar", figsize=(11, 5))
plt.xticks(rotation=25, ha="right"); plt.ylim(0, 1.05)
plt.title("Tuned Models: Key Metrics")
plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

## 3. ROC and PR Curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
for name, d in store.items():
    fpr, tpr, _ = roc_curve(y_test, d["score"])
    prec, rec, _ = precision_recall_curve(y_test, d["score"])
    auc_ = results.loc[f"{name} (tuned)", "ROC-AUC"]
    ap_ = results.loc[f"{name} (tuned)", "PR-AUC"]
    ax[0].plot(fpr, tpr, label=f"{name} (AUC={auc_:.3f})")
    ax[1].plot(rec, prec, label=f"{name} (AP={ap_:.3f})")
ax[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[0].set_title("ROC Curves"); ax[0].legend()
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision")
ax[1].set_title("PR Curves"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. Best Model

In [ ]:
best_name = tuned_only["PR-AUC"].idxmax()
b = tuned_only.loc[best_name]
print(f"Best model: {best_name}")
print(f"  Recall={b['Recall']:.3f}  Precision={b['Precision']:.3f}  F1={b['F1']:.3f}  PR-AUC={b['PR-AUC']:.3f}")
print(f"  Frauds caught: {int(b['TP'])} of {int(b['TP'] + b['FN'])} | False alarms: {int(b['FP'])}")

In [ ]:
base = store[best_name.replace(" (tuned)", "")]
cm = confusion_matrix(y_test, base["pred"])
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
plt.title(f"Best Model: {best_name}")
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.show()